# BRYAN YUNGA JUMBO
# Ejercicio 10: Re-ranking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

## Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

In [1]:
!pip install beir

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 18.7 MB/s eta 0:00:00


In [2]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd

/usr/local/lib/python3.12/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [3]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

../data/beir_datasets/scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

'../data/beir_datasets/scifact'

In [4]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

  0%|          | 0/5183 [00:00<?, ?it/s]

In [5]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [6]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [7]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [8]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


In [9]:
!pip install rank_bm25

## Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

In [10]:
import re

def tokenize(text):
    # Tokenización simple: minúsculas y solo palabras alfanuméricas
    return re.findall(r"\w+", text.lower())

# Mantenemos el orden de doc_ids para poder mapear índice -> doc_id después
doc_ids = list(corpus.keys())
tokenized_corpus = [
    tokenize(corpus[doc_id].get("title", "") + " " + corpus[doc_id].get("text", ""))
    for doc_id in doc_ids
]

**Construir Indice BM25**

In [11]:
from rank_bm25 import BM25Okapi

bm25 = BM25Okapi(tokenized_corpus)

**Retrieval con BM25 para todas las queries**

In [12]:
from tqdm import tqdm

TOP_K = 100  # recuperamos más de 10 para luego re-rankear sobre ese conjunto

run_bm25 = {}

for qid, query_text in tqdm(queries.items()):
    query_tokens = tokenize(query_text)
    scores = bm25.get_scores(query_tokens)
    top_indices = scores.argsort()[::-1][:TOP_K]
    run_bm25[qid] = {
        doc_ids[idx]: float(scores[idx])
        for idx in top_indices
    }

100%|██████████| 300/300 [00:11<00:00, 25.20it/s]


**Métricas Recall@10 y nDCG@10**

In [13]:
from beir.retrieval.evaluation import EvaluateRetrieval

k_values = [1, 5, 10, 100]
ndcg_base, map_base, recall_base, precision_base = EvaluateRetrieval.evaluate(qrels, run_bm25, k_values)

print("=== Baseline BM25 ===")
print("nDCG@10:", ndcg_base["NDCG@10"])
print("Recall@10:", recall_base["Recall@10"])

=== Baseline BM25 ===
nDCG@10: 0.65189
Recall@10: 0.774


## Parte 3. Implementación del re-ranking _cross-encoder_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

**Instalar sentence-transformers**

In [14]:
!pip install sentence-transformers

**Cargar el modelo cross-encoder**

In [15]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

**Re-rankear los top-k de cada query**

In [16]:
run_cross_encoder = {}

for qid, doc_scores in tqdm(run_bm25.items()):
    query_text = queries[qid]
    candidate_doc_ids = list(doc_scores.keys())

    pairs = [
        (query_text, corpus[doc_id].get("title", "") + " " + corpus[doc_id].get("text", ""))
        for doc_id in candidate_doc_ids
    ]

    ce_scores = cross_encoder.predict(pairs)

    run_cross_encoder[qid] = {
        doc_id: float(score)
        for doc_id, score in zip(candidate_doc_ids, ce_scores)
    }

100%|██████████| 300/300 [02:31<00:00,  1.98it/s]


**Identificar cambios de posición en el top 10**

In [17]:
def get_top_k_ids(run_dict, qid, k=10):
    sorted_docs = sorted(run_dict[qid].items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, score in sorted_docs[:k]]

cambios = []

for qid in queries.keys():
    top10_bm25 = get_top_k_ids(run_bm25, qid, 10)
    top10_ce = get_top_k_ids(run_cross_encoder, qid, 10)

    if top10_bm25 != top10_ce:
        cambios.append({
            "query_id": qid,
            "top10_bm25": top10_bm25,
            "top10_cross_encoder": top10_ce
        })

print(f"Queries cuyo top-10 cambió: {len(cambios)} de {len(queries)}")

# Ejemplo de una query con cambios
if cambios:
    ejemplo = cambios[0]
    print("\nQuery:", ejemplo["query_id"])
    print("Top-10 BM25:           ", ejemplo["top10_bm25"])
    print("Top-10 Cross-Encoder:  ", ejemplo["top10_cross_encoder"])

Queries cuyo top-10 cambió: 300 de 300

Query: 1
Top-10 BM25:            ['10608397', '43385013', '40212412', '10931595', '27049238', '803312', '10906636', '13231899', '393001', '3770726']
Top-10 Cross-Encoder:   ['43385013', '121581019', '6863070', '4459491', '393001', '36637129', '35621259', '10906636', '1156322', '4983']


## Parte 4. Implementación del re-ranking _LTR_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

**Instalar LightGBM**

In [18]:
!pip install lightgbm

**Construir las features por par (query, doc)**

In [19]:
import numpy as np

def build_features(qid, doc_id, query_text):
    doc = corpus[doc_id]
    doc_text = doc.get("title", "") + " " + doc.get("text", "")

    bm25_score = run_bm25[qid].get(doc_id, 0.0)
    ce_score = run_cross_encoder[qid].get(doc_id, 0.0)
    doc_len = len(doc_text.split())

    query_terms = set(tokenize(query_text))
    doc_terms = set(tokenize(doc_text))
    overlap = len(query_terms & doc_terms)
    overlap_ratio = overlap / len(query_terms) if len(query_terms) > 0 else 0.0

    return [bm25_score, ce_score, doc_len, overlap, overlap_ratio]

feature_names = ["bm25_score", "ce_score", "doc_len", "term_overlap", "overlap_ratio"]

**Armar el dataset de entrenamiento (X, y, groups)**

In [20]:
X_rows = []
y_rows = []
group_sizes = []
row_qid_docid = []  # para poder recuperar después qué fila es qué (qid, doc_id)

for qid in queries.keys():
    if qid not in qrels:
        continue

    query_text = queries[qid]
    candidate_doc_ids = list(run_bm25[qid].keys())

    group_count = 0
    for doc_id in candidate_doc_ids:
        feats = build_features(qid, doc_id, query_text)
        label = qrels[qid].get(doc_id, 0)  # 0 si no es relevante

        X_rows.append(feats)
        y_rows.append(label)
        row_qid_docid.append((qid, doc_id))
        group_count += 1

    group_sizes.append(group_count)

X = np.array(X_rows)
y = np.array(y_rows)

print("Shape X:", X.shape)
print("Total queries en grupos:", len(group_sizes))
print("Documentos relevantes en y:", (y > 0).sum())

Shape X: (30000, 5)
Total queries en grupos: 300
Documentos relevantes en y: 294


**Entrenar el modelo LambdaRank**

In [21]:
import lightgbm as lgb

train_data = lgb.Dataset(X, label=y, group=group_sizes)

params = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [10],
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 5,
    "verbose": -1
}

ltr_model = lgb.train(
    params,
    train_data,
    num_boost_round=100
)

**Re-rankear con el modelo LTR**

In [22]:
run_ltr = {}

for qid in queries.keys():
    if qid not in qrels:
        continue

    query_text = queries[qid]
    candidate_doc_ids = list(run_bm25[qid].keys())

    X_query = np.array([
        build_features(qid, doc_id, query_text)
        for doc_id in candidate_doc_ids
    ])

    scores = ltr_model.predict(X_query)

    run_ltr[qid] = {
        doc_id: float(score)
        for doc_id, score in zip(candidate_doc_ids, scores)
    }

**Identificar cambios de posición en el top 10**

In [23]:
cambios_ltr = []

for qid in run_ltr.keys():
    top10_bm25 = get_top_k_ids(run_bm25, qid, 10)
    top10_ltr = get_top_k_ids(run_ltr, qid, 10)

    if top10_bm25 != top10_ltr:
        cambios_ltr.append({
            "query_id": qid,
            "top10_bm25": top10_bm25,
            "top10_ltr": top10_ltr
        })

print(f"Queries cuyo top-10 cambió (LTR vs BM25): {len(cambios_ltr)} de {len(run_ltr)}")

if cambios_ltr:
    ejemplo = cambios_ltr[0]
    print("\nQuery:", ejemplo["query_id"])
    print("Top-10 BM25:", ejemplo["top10_bm25"])
    print("Top-10 LTR:  ", ejemplo["top10_ltr"])

Queries cuyo top-10 cambió (LTR vs BM25): 300 de 300

Query: 1
Top-10 BM25: ['10608397', '43385013', '40212412', '10931595', '27049238', '803312', '10906636', '13231899', '393001', '3770726']
Top-10 LTR:   ['43385013', '8509018', '31439189', '10608397', '121581019', '393001', '27049238', '24660385', '1156322', '10906636']


## Parte 5. Evaluación post re-ranking

Calcular métricas:
* nDCG@10
* MAP
* Recall@10

**Calcular nDCG@10, MAP y Recall@10 para ambos métodos**

In [24]:
k_values = [1, 5, 10, 100]

# Cross-encoder
ndcg_ce, map_ce, recall_ce, precision_ce = EvaluateRetrieval.evaluate(qrels, run_cross_encoder, k_values)

# LTR
ndcg_ltr, map_ltr, recall_ltr, precision_ltr = EvaluateRetrieval.evaluate(qrels, run_ltr, k_values)

print("=== BM25 (baseline) ===")
print("nDCG@10:", ndcg_base["NDCG@10"], "| MAP@100:", map_base["MAP@100"], "| Recall@10:", recall_base["Recall@10"])

print("\n=== Cross-Encoder ===")
print("nDCG@10:", ndcg_ce["NDCG@10"], "| MAP@100:", map_ce["MAP@100"], "| Recall@10:", recall_ce["Recall@10"])

print("\n=== LTR (LightGBM) ===")
print("nDCG@10:", ndcg_ltr["NDCG@10"], "| MAP@100:", map_ltr["MAP@100"], "| Recall@10:", recall_ltr["Recall@10"])

=== BM25 (baseline) ===
nDCG@10: 0.65189 | MAP@100: 0.61318 | Recall@10: 0.774

=== Cross-Encoder ===
nDCG@10: 0.67545 | MAP@100: 0.63731 | Recall@10: 0.79061

=== LTR (LightGBM) ===
nDCG@10: 0.86939 | MAP@100: 0.86334 | Recall@10: 0.87306


**Tabla comparativa final**

In [25]:
df_comparacion = pd.DataFrame({
    "Método": ["BM25 (baseline)", "Cross-Encoder", "LTR (LightGBM)"],
    "nDCG@10": [ndcg_base["NDCG@10"], ndcg_ce["NDCG@10"], ndcg_ltr["NDCG@10"]],
    "MAP@100": [map_base["MAP@100"], map_ce["MAP@100"], map_ltr["MAP@100"]],
    "Recall@10": [recall_base["Recall@10"], recall_ce["Recall@10"], recall_ltr["Recall@10"]],
})

df_comparacion

,Método,nDCG@10,MAP@100,Recall@10
0,BM25 (baseline),0.65189,0.61318,0.77400
1,Cross-Encoder,0.67545,0.63731,0.79061
2,LTR (LightGBM),0.86939,0.86334,0.87306
